In [1]:
import os
import sys
import argparse
import jax
import jax.numpy as jnp
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from functools import partial

# JAX parallelization imports
from jax.experimental import mesh_utils
from jax.sharding import Mesh, PartitionSpec, NamedSharding

# Set CUDA devices (adjust based on your available GPUs)
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"  # Adjust for your GPU setup

# Configure JAX
jax.config.update('jax_platform_name', 'gpu')
from jax.lib import xla_bridge
print(f"JAX version: {jax.__version__}")
print(f"Platform: {xla_bridge.get_backend().platform}")

# Local imports
from config_generator import generate_default_config, load_config, save_config
from power_spectrum import setup_power_spectrum
from data_loader import setup_data_loaders
from muse_problem import create_muse_problem
from optimization import MuseOptimizer, CovarianceEstimator, run_full_analysis

JAX version: 0.4.26
Platform: gpu


In [2]:
!nvidia-smi


Tue Jun 17 15:27:50 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 530.30.02              Driver Version: 530.30.02    CUDA Version: 12.1     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                  Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf            Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti      Off| 00000000:1A:00.0 Off |                  N/A |
| 16%   36C    P2               51W / 250W|   2051MiB / 11264MiB |      6%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

/home/guoqi/.conda/envs/MAPLE/lib/python3.9/pty.py:85: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


|   4  Tesla V100-PCIE-32GB            Off| 00000000:21:00.0 Off |                    0 |
| N/A   32C    P0               36W / 250W|   7760MiB / 32768MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+
|   5  Tesla V100-PCIE-32GB            Off| 00000000:22:00.0 Off |                    0 |
| N/A   32C    P0               35W / 250W|  25312MiB / 32768MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+
|   6  NVIDIA GeForce RTX 2080 Ti      Off| 00000000:23:00.0 Off |                  N/A |
| 16%   28C    P8               17W / 250W|      1MiB / 11264MiB |      0%      Default |
|                                         |                      |                  N/A |
+---------

In [3]:
def setup_parallel_environment():
    """Setup the parallel computational environment"""
    print("Setting up parallel environment...")
    print(f"JAX version: {jax.__version__}")
    print(f"Platform: {xla_bridge.get_backend().platform}")
    
    # Get available devices
    devices = jax.devices()
    n_devices = len(devices)
    print(f"Available devices: {devices}")
    print(f"Number of devices: {n_devices}")
    
    if n_devices < 2:
        print("Warning: Only one device available. Parallelization will be limited.")
    
    # Create mesh for parallelization
    # Using a 1D mesh across devices for different starting points
    mesh = Mesh(mesh_utils.create_device_mesh((n_devices,)), ("start_points",))
    
    return mesh, n_devices

In [ ]:
def generate_starting_points(problem, n_starting_points, rng_key, config):
    """
    Generate multiple starting points for the optimization
    
    Args:
        problem: The MUSE problem instance
        n_starting_points: Number of different starting points to generate
        rng_key: JAX random key
        config: Configuration dictionary
        
    Returns:
        Array of starting points with shape (n_starting_points, n_params)
    """
    print(f"Generating {n_starting_points} starting points...")
    
    # Get the default starting point
    default_start = problem.get_starting_point()
    n_params = len(default_start)
    
    # Generate random keys for each starting point
    rng_keys = jax.random.split(rng_key, n_starting_points)
    
    def generate_single_start(rng_key, default_start):
        """Generate a single perturbed starting point"""
        # Add random perturbations to the default starting point
        # Adjust perturbation scale based on your problem
        perturbation_scale = 0.1
        
        # Generate random perturbations
        perturbations = jax.random.normal(rng_key, shape=default_start.shape) * perturbation_scale
        
        # Apply perturbations (you might want to customize this based on parameter constraints)
        perturbed_start = default_start * (1.0 + perturbations)
        
        return perturbed_start
    
    # Vectorize the function to generate all starting points
    generate_starts_vmap = jax.vmap(generate_single_start, in_axes=(0, None))
    starting_points = generate_starts_vmap(rng_keys, default_start)
    
    return starting_points


In [ ]:

@partial(jax.jit, static_argnums=(0, 1))
def run_single_optimization(problem, config, x_data, starting_point, rng):
    """
    Run optimization for a single starting point (JIT compiled for efficiency)
    
    Args:
        problem: The MUSE problem instance
        config: Configuration dictionary
        x_data: Observation data
        starting_point: Starting point for optimization
        rng_key: Random key for this optimization
        
    Returns:
        Optimization results for this starting point
    """
    # # Create optimizer
    # optimizer = MuseOptimizer(problem, config)
    
    # # Run optimization
    # opt_results = optimizer.run_optimization(x_data, starting_point, rng_key)
    result = run_full_analysis(problem, config, x_data, start_point, rng)
    
    return opt_results

In [ ]:
def run_parallel_optimizations(problem, config, x_data, starting_points, rng_keys, mesh):
    """
    Run optimizations in parallel across multiple starting points
    
    Args:
        problem: The MUSE problem instance
        config: Configuration dictionary
        x_data: Observation data
        starting_points: Array of starting points (n_starts, n_params)
        rng_keys: Array of random keys for each optimization
        mesh: JAX mesh for parallelization
        
    Returns:
        List of optimization results for each starting point
    """
    print(f"Running {len(starting_points)} optimizations in parallel...")
    
    # Create sharding for parallel execution
    sharding = NamedSharding(mesh, PartitionSpec("start_points"))
    
    # Shard the starting points and keys across devices
    starting_points_sharded = jax.lax.with_sharding_constraint(starting_points, sharding)
    rng_keys_sharded = jax.lax.with_sharding_constraint(rng_keys, sharding)
    
    # Vectorize the optimization function across starting points
    run_optimizations_vmap = jax.vmap(
        run_single_optimization, 
        in_axes=(None, None, None, 0, 0)  # Vectorize over starting_points and rng_keys
    )
    
    # Run all optimizations in parallel
    all_results = run_optimizations_vmap(
        problem, config, x_data, starting_points_sharded, rng_keys_sharded
    )
    
    return all_results


In [ ]:

def run_parallel_p3d_analysis(config, mesh):
    """
    Run the complete parallelized P3D MUSE analysis
    
    Args:
        config: Configuration dictionary
        mesh: JAX mesh for parallelization
        
    Returns:
        Dictionary containing analysis results
    """
    print("Starting parallelized P3D MUSE analysis...")
    start_time = datetime.now()
    
    # Setup power spectrum
    print("Setting up power spectrum...")
    power_data = setup_power_spectrum(config)
    
    # Load data
    print("Loading data...")
    data_loaders = setup_data_loaders(config)
    
    # Create MUSE problem
    print("Creating MUSE problem...")
    problem = create_muse_problem(power_data, data_loaders, config)
    problem.set_x(data_loaders['lya']['map_lya_sim'])
    
    # Initial conditions 
    print("Initial conditions...")
    key = jax.random.PRNGKey(100)
    x_data = problem.sample_x_z(key, problem.tf_cut_flat)[0]
    
    # Setup random number generator
    rng = jax.random.PRNGKey(config['random_seed'])
    
    # Generate multiple starting points
    n_starting_points = 10 # config['parallelization']['n_starting_points'] # could add to config in the future
    starting_points = generate_starting_points(problem, n_starting_points, rng, config)
    
    # Generate random keys for each optimization
    rng_keys = jax.random.split(rng, n_starting_points)
    
    # Run parallel optimizations
    print(f"Running {n_starting_points} optimizations in parallel...")
    all_opt_results = run_parallel_optimizations(
        problem, config, x_data, starting_points, rng_keys, mesh
    )
            
    print(f"Parallel analysis completed in {results['runtime']}")
    
    return results


In [ ]:

def save_parallel_results(results, config):
    """Save parallelized analysis results"""
    output_dir = config['output']['output_dir']
    
    # Save main results
    np.save(os.path.join(output_dir, 'final_parameters.npy'), results['final_parameters'])
    np.save(os.path.join(output_dir, 'fiducial_parameters.npy'), results['fiducial_parameters'])
    np.save(os.path.join(output_dir, 'all_final_losses.npy'), results['all_final_losses'])
    np.save(os.path.join(output_dir, 'starting_points.npy'), results['starting_points'])
    np.save(os.path.join(output_dir, 'best_starting_point_idx.npy'), results['best_starting_point_idx'])
    
    if results['parameter_covariance'] is not None:
        np.save(os.path.join(output_dir, 'parameter_covariance.npy'), results['parameter_covariance'])
    
    # Save optimization history for best result
    if 'history' in results['best_optimization']:
        np.save(os.path.join(output_dir, 'best_optimization_history.npy'), 
                results['best_optimization']['history'])
    
    # Save all results if requested
    if results['all_optimization_results'] is not None:
        for i, opt_result in enumerate(results['all_optimization_results']):
            np.save(os.path.join(output_dir, f'optimization_result_{i}.npy'), opt_result)
    
    # Save covariance components
    if results['covariance'] is not None:
        if 'J' in results['covariance']:
            np.save(os.path.join(output_dir, 'score_covariance_J.npy'), results['covariance']['J'])
        if 'H' in results['covariance']:
            np.save(os.path.join(output_dir, 'fisher_matrix_H.npy'), results['covariance']['H'])
    
    # Create summary plot
    if config['output']['plot_progress']:
        create_parallel_summary_plot(results, output_dir)
    
    print(f"Parallel results saved to {output_dir}")



In [ ]:


def main():
    """Main analysis function with parallelization"""
    parser = argparse.ArgumentParser(description='Parallelized P3D MUSE Analysis')
    parser.add_argument('--config', type=str, default=None,
                        help='Path to configuration file')
    parser.add_argument('--output-dir', type=str, default='./results/',
                        help='Output directory for results')
    parser.add_argument('--maxsteps', type=int, default=None,
                        help='Maximum optimization steps')
    parser.add_argument('--nsims', type=int, default=None,
                        help='Number of simulations for MUSE')
    parser.add_argument('--n-starts', type=int, default=None,
                        help='Number of starting points to try')
    parser.add_argument('--no-plot', action='store_true',
                        help='Disable plotting')
    parser.add_argument('--test-mode', action='store_true',
                        help='Run in test mode with reduced parameters')
    parser.add_argument('--save-all', action='store_true',
                        help='Save results from all starting points')
    
    args = parser.parse_args()
    
    # Setup parallel environment
    mesh, n_devices = setup_parallel_environment()
    
    # Load or create configuration
    if args.config:
        try:
            config = load_config(args.config)
            print(f"Loaded configuration from {args.config}")
        except FileNotFoundError:
            print(f"Configuration file {args.config} not found. Creating default config.")
            config = create_default_config()
            save_config(config, args.config)
    else:
        config = create_default_config()
    
    # Override config with command line arguments
    if args.output_dir:
        config['output']['output_dir'] = args.output_dir
    if args.maxsteps:
        config['optimization']['maxsteps'] = args.maxsteps
    if args.nsims:
        config['optimization']['nsims'] = args.nsims
    if args.n_starts:
        config['parallelization']['n_starting_points'] = args.n_starts
    if args.no_plot:
        config['output']['plot_progress'] = False
    if args.save_all:
        config['output']['save_all_results'] = True
    
    # Adjust number of starting points based on available devices
    n_starting_points = config['parallelization']['n_starting_points']
    if n_starting_points > n_devices:
        print(f"Note: {n_starting_points} starting points requested, but only {n_devices} devices available.")
        print("Optimizations will be batched across devices.")
    
    # Test mode adjustments
    if args.test_mode:
        print("Running in test mode with reduced parameters...")
        config['optimization']['maxsteps'] = 10
        config['optimization']['nsims'] = 3
        config['covariance']['nsims_score'] = 10
        config['covariance']['nsims_fisher'] = 3
        config['grid']['nc'] = 64  # Smaller grid for faster testing
        config['parallelization']['n_starting_points'] = min(4, n_devices)
    
    # Create output directory
    os.makedirs(config['output']['output_dir'], exist_ok=True)
    
    # Save configuration
    config_file = os.path.join(config['output']['output_dir'], 'config.yaml')
    save_config(config, config_file)
    print(f"Configuration saved to {config_file}")
    
    try:
        # Run the parallelized analysis
        results = run_parallel_p3d_analysis(config, mesh)
        
        # Save results
        if config['output']['save_results']:
            save_parallel_results(results, config)
        
        print("Parallel analysis completed successfully!")
        
    except Exception as e:
        print(f"Error during analysis: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)
